In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

M = '/content/drive/MyDrive/mimic-iv/hosp/'
model_df = pd.read_csv('/content/drive/MyDrive/mimic-iv/aortic_aneurysm_final_modeling_dataset.csv')

adm = pd.read_csv(M + 'admissions.csv.gz', usecols=['hadm_id', 'admission_type'])
model_df = model_df.merge(adm, on='hadm_id', how='left')

print(len(model_df))  # must be 780
print(model_df['admission_type'].value_counts(dropna=False))
print(pd.crosstab(model_df['admission_type'], model_df['approach'], margins=True))
print(model_df.groupby('admission_type')['aki'].agg(['mean', 'count']))

In [ ]:
d_diag = pd.read_csv(M + 'd_icd_diagnoses.csv.gz', dtype={'icd_code': str})
t = d_diag['long_title'].str.lower()
mask = t.str.contains('aneurysm') & t.str.contains('ruptur')
print(d_diag[mask].sort_values(['icd_version', 'icd_code']).to_string(index=False))

In [ ]:
import statsmodels.formula.api as smf

# --- rupture flag (codes discovered from dictionary, not typed) ---
d_diag = pd.read_csv(M + 'd_icd_diagnoses.csv.gz', dtype={'icd_code': str})
t = d_diag['long_title'].str.lower()
rup_codes = d_diag[t.str.contains('aneurysm') & t.str.contains('ruptured')
                   & ~t.str.contains('nonruptured') & ~t.str.contains('cerebral')]
print(len(rup_codes), "ruptured aortic codes (expect 19)")
print(rup_codes[['icd_code', 'icd_version', 'long_title']].to_string(index=False))

diag = pd.read_csv(M + 'diagnoses_icd.csv.gz', dtype={'icd_code': str},
                   usecols=['hadm_id', 'icd_code', 'icd_version'])
diag = diag[diag['hadm_id'].isin(model_df['hadm_id'])]
rup_hadm = diag.merge(rup_codes[['icd_code', 'icd_version']],
                      on=['icd_code', 'icd_version'])['hadm_id'].unique()
model_df['rupture'] = model_df['hadm_id'].isin(rup_hadm).astype(int)

# --- urgency (3 levels; observation kept separate on purpose) ---
def urgency(x):
    if x in ['ELECTIVE', 'SURGICAL SAME DAY ADMISSION']:
        return 'elective'
    if x in ['EW EMER.', 'DIRECT EMER.', 'URGENT']:
        return 'emergency_urgent'
    return 'observation'
model_df['urgency'] = model_df['admission_type'].apply(urgency)

print(len(model_df))  # must still be 780
print(model_df['rupture'].value_counts())
print(pd.crosstab(model_df['rupture'], model_df['approach'], margins=True))
print(model_df.groupby('rupture')['aki'].agg(['mean', 'count']))
print(pd.crosstab(model_df['rupture'], model_df['urgency']))

# --- four models; track only the approach (open vs endovascular) effect ---
base = ('aki ~ C(approach, Treatment(reference="endovascular")) '
        '+ C(location, Treatment(reference="abdominal")) '
        '+ age_at_admission + C(gender) + cci + vasopressor + C(anchor_year_group)')

def fit(formula, data, label):
    m = smf.logit(formula, data=data).fit(disp=0)
    term = [p for p in m.params.index if 'approach' in p][0]
    ci = m.conf_int().loc[term]
    return {'model': label, 'N': int(m.nobs), 'events': int(data['aki'].sum()),
            'OR_open': np.exp(m.params[term]), 'CI_low': np.exp(ci[0]),
            'CI_high': np.exp(ci[1]), 'p': m.pvalues[term]}

res = [
    fit(base, model_df, 'A. Original (check: OR ~1.87, N=780)'),
    fit(base + ' + C(urgency, Treatment(reference="elective"))', model_df, 'B. + urgency'),
    fit(base + ' + rupture', model_df, 'C. + rupture flag'),
    fit(base, model_df[model_df['rupture'] == 0], 'D. Excluding ruptured'),
]
print(pd.DataFrame(res).round(3).to_string(index=False))

model_df.to_csv('/content/drive/MyDrive/mimic-iv/aortic_aneurysm_final_modeling_dataset_v2.csv', index=False)

In [ ]:
mC = smf.logit(base + ' + rupture', data=model_df).fit(disp=0)
print(mC.summary())

orC = pd.DataFrame({'OR': np.exp(mC.params),
                    'CI_low': np.exp(mC.conf_int()[0]),
                    'CI_high': np.exp(mC.conf_int()[1]),
                    'p': mC.pvalues}).round(3)
print(orC.to_string())

import patsy
from statsmodels.stats.outliers_influence import variance_inflation_factor
y, X = patsy.dmatrices(base + ' + rupture', data=model_df, return_type='dataframe')
vif = pd.DataFrame({'variable': X.columns,
                    'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]})
print(vif.to_string(index=False))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

EICU_PATH = '/content/drive/MyDrive/eICU-CRD/'
eicu = pd.read_csv(EICU_PATH + 'aortic_aneurysm_eicu_cohort_final.csv')

dx = pd.read_csv(EICU_PATH + 'admissionDx.csv.gz', usecols=['patientunitstayid', 'admitdxtext'])
dx = dx[dx['patientunitstayid'].isin(eicu['patientunitstayid'])]
rup_stays = dx[dx['admitdxtext'].str.contains('with rupture', case=False, na=False)]['patientunitstayid'].unique()
eicu['rupture'] = eicu['patientunitstayid'].isin(rup_stays).astype(int)

m = eicu.dropna(subset=['aki']).copy()
print(len(eicu), len(m))  # expect 395 and 384
print(m['rupture'].value_counts())
print(m.groupby('rupture')['aki'].agg(['mean', 'count']))

f = ('aki ~ C(location, Treatment(reference="abdominal")) + age_numeric '
     '+ C(gender) + cci + vasopressor + rupture')
mod = smf.logit(f, data=m).fit(disp=0)
print(mod.summary())
print(pd.DataFrame({'OR': np.exp(mod.params),
                    'CI_low': np.exp(mod.conf_int()[0]),
                    'CI_high': np.exp(mod.conf_int()[1]),
                    'p': mod.pvalues}).round(3).to_string())

In [ ]:
# save eICU data with rupture so it isn't lost
m.to_csv(EICU_PATH + 'aortic_aneurysm_eicu_cohort_final_v2.csv', index=False)

mim = pd.read_csv('/content/drive/MyDrive/mimic-iv/aortic_aneurysm_final_modeling_dataset_v2.csv')
mim = mim[mim['location'].isin(['abdominal', 'thoracic'])].copy()  # thoracoabdominal not in eICU
mim['male'] = (mim['gender'] == 'M').astype(int)
mim['age'] = mim['age_at_admission']
mim['cohort'] = 'MIMIC'

ei = m.copy()
ei['male'] = (ei['gender'] == 'Male').astype(int)
ei['age'] = ei['age_numeric']
ei['cohort'] = 'eICU'

print(len(mim), len(ei))  # expect 731 and 384

# matched MIMIC-IV model: same covariates as eICU, no approach/era/thoracoabdominal
f = 'aki ~ C(location, Treatment(reference="abdominal")) + age + male + cci + vasopressor + rupture'
mm = smf.logit(f, data=mim).fit(disp=0)
print(pd.DataFrame({'OR': np.exp(mm.params),
                    'CI_low': np.exp(mm.conf_int()[0]),
                    'CI_high': np.exp(mm.conf_int()[1]),
                    'p': mm.pvalues}).round(3).to_string())

# pooled model: does the location effect differ by cohort?
cols = ['aki', 'location', 'age', 'male', 'cci', 'vasopressor', 'rupture', 'cohort']
pool = pd.concat([mim[cols], ei[cols]], ignore_index=True)
fp = ('aki ~ C(location, Treatment(reference="abdominal")) * C(cohort, Treatment(reference="MIMIC")) '
      '+ age + male + cci + vasopressor + rupture')
mp = smf.logit(fp, data=pool).fit(disp=0)
print(mp.summary().tables[1])

In [ ]:
tr = pd.read_csv(EICU_PATH + 'treatment.csv.gz', usecols=['patientunitstayid', 'treatmentstring'])
tr = tr[tr['patientunitstayid'].isin(m['patientunitstayid'])]
tr = tr.merge(m[['patientunitstayid', 'location']], on='patientunitstayid')
tr['top'] = tr['treatmentstring'].str.split('|').str[:2].str.join('|')

n_stays = m.groupby('location')['patientunitstayid'].nunique()
share = (tr.drop_duplicates(['patientunitstayid', 'top'])
           .groupby(['top', 'location'])['patientunitstayid'].nunique()
           .unstack(fill_value=0))
share_pct = (share / n_stays * 100).round(1)
share_pct['diff'] = (share_pct['thoracic'] - share_pct['abdominal']).abs()
print(share_pct.sort_values('diff', ascending=False).head(25))

In [ ]:
cs = tr[tr['treatmentstring'].str.startswith('cardiovascular|cardiac surgery')]['patientunitstayid'].unique()
m['cardiac_surgery_tx'] = m['patientunitstayid'].isin(cs).astype(int)
print(pd.crosstab(m['location'], m['cardiac_surgery_tx']))
print(m.groupby(['location', 'cardiac_surgery_tx'])['aki'].agg(['mean', 'count']))

m2 = m[m['cardiac_surgery_tx'] == 0].copy()
print(len(m2), int(m2['aki'].sum()), "stays / events after exclusion")

f = 'aki ~ C(location, Treatment(reference="abdominal")) + age_numeric + C(gender) + cci + vasopressor + rupture'
mod2 = smf.logit(f, data=m2).fit(disp=0)
print(pd.DataFrame({'OR': np.exp(mod2.params),
                    'CI_low': np.exp(mod2.conf_int()[0]),
                    'CI_high': np.exp(mod2.conf_int()[1]),
                    'p': mod2.pvalues}).round(3).to_string())

# re-run the pooled interaction test with the restricted eICU cohort
ei2 = m2.copy()
ei2['male'] = (ei2['gender'] == 'Male').astype(int)
ei2['age'] = ei2['age_numeric']
ei2['cohort'] = 'eICU'
pool2 = pd.concat([mim[cols], ei2[cols]], ignore_index=True)
mp2 = smf.logit(fp, data=pool2).fit(disp=0)
print(mp2.summary().tables[1])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

mim = pd.read_csv('/content/drive/MyDrive/mimic-iv/aortic_aneurysm_final_modeling_dataset_v2.csv')
ei = pd.read_csv('/content/drive/MyDrive/eICU-CRD/aortic_aneurysm_eicu_cohort_final_v2.csv')

for c in ['aki', 'approach', 'location', 'anchor_year_group', 'age_at_admission', 'gender',
          'cci', 'vasopressor', 'rupture', 'urgency']:
    assert c in mim.columns, f'MIMIC file missing {c}'
for c in ['aki', 'location', 'age_numeric', 'gender', 'cci', 'vasopressor', 'rupture',
          'has_diagnosis_data']:
    assert c in ei.columns, f'eICU file missing {c}'
print(len(mim), len(ei))  # expect 780 and 384

NA = 'Not available'

def npct(mask):
    return f'{int(mask.sum())} ({100 * mask.mean():.1f}%)'

def mean_sd(s):
    return f'{s.mean():.1f} ({s.std():.1f})'

def med_iqr(s):
    return f'{s.median():.0f} [{s.quantile(.25):.0f}-{s.quantile(.75):.0f}]'

rows = []
def add(label, a, b):
    rows.append((label, a, b))

add('N', str(len(mim)), str(len(ei)))
add('Age, years, mean (SD)', mean_sd(mim['age_at_admission']), mean_sd(ei['age_numeric']))
add('Age, years, median [IQR]', med_iqr(mim['age_at_admission']), med_iqr(ei['age_numeric']))
add('Male sex, n (%)', npct(mim['gender'] == 'M'), npct(ei['gender'] == 'Male'))

add('Aneurysm location, n (%)', '', '')
for lev in ['abdominal', 'thoracic', 'thoracoabdominal']:
    add(f'  {lev.capitalize()}', npct(mim['location'] == lev),
        npct(ei['location'] == lev) if lev != 'thoracoabdominal' else NA)

add('Repair approach, n (%)', '', '')
for lev in ['endovascular', 'open']:
    add(f'  {lev.capitalize()}', npct(mim['approach'] == lev), NA)

add('Ruptured aneurysm, n (%)', npct(mim['rupture'] == 1), npct(ei['rupture'] == 1))
add('Charlson index, mean (SD)', mean_sd(mim['cci']), mean_sd(ei['cci']))
add('Charlson index, median [IQR]', med_iqr(mim['cci']), med_iqr(ei['cci']))
add('Diagnosis data available, n (%)', 'All', npct(ei['has_diagnosis_data'] == 1))
add('Vasopressor use, n (%)', npct(mim['vasopressor'] == 1), npct(ei['vasopressor'] == 1))
add('Acute kidney injury, n (%)', npct(mim['aki'] == 1), npct(ei['aki'] == 1))

add('Admission type, n (%)', '', '')
for lev in ['elective', 'emergency_urgent', 'observation']:
    add(f'  {lev.replace("_", "/").capitalize()}', npct(mim['urgency'] == lev), NA)

add('Era (anchor year group), n (%)', '', '')
for lev in sorted(mim['anchor_year_group'].unique()):
    add(f'  {lev}', npct(mim['anchor_year_group'] == lev), NA)

table1 = pd.DataFrame(rows, columns=['Characteristic', 'MIMIC-IV', 'eICU-CRD'])
print(table1.to_string(index=False))
table1.to_csv('/content/drive/MyDrive/table1_aortic_aneurysm.csv', index=False)

In [ ]:
import pandas as pd
d_diag = pd.read_csv(M + 'd_icd_diagnoses.csv.gz')
pat = r'end stage renal|end-stage renal|dependence on renal dialysis|dialysis status|renal dialysis status|encounter for.*dialysis'
cand = d_diag[d_diag['long_title'].str.contains(pat, case=False, regex=True)]
print(cand[['icd_code','icd_version','long_title']].to_string())

In [ ]:
core = ['5856','N186','V451','V4511','Z992','40301','40311','40391','40402','40403',
        '40412','40413','40492','40493','I120','I1311','I132']
broad = core + ['V560','V568','Z49','Z4901','Z4902']

cohort = pd.read_csv('/content/drive/MyDrive/mimic-iv/aortic_aneurysm_final_modeling_dataset_v2.csv')
diag = pd.read_csv(M + 'diagnoses_icd.csv.gz', dtype={'icd_code': str})
d = diag[diag['hadm_id'].isin(cohort['hadm_id'])]

for name, codes in [('core', core), ('broad', broad)]:
    ids = d.loc[d['icd_code'].str.strip().isin(codes), 'hadm_id'].unique()
    x = cohort.assign(esrd=cohort['hadm_id'].isin(ids))
    print(f"\n{name}: {len(ids)} of {len(cohort)}")
    print(x.groupby('esrd')['aki'].agg(['count','sum','mean']))
    print(x.groupby('esrd')['approach'].value_counts().unstack())

In [ ]:
import numpy as np, statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

ids = d.loc[d['icd_code'].str.strip().isin(core), 'hadm_id'].unique()
c3 = cohort[~cohort['hadm_id'].isin(ids)].copy()
print("N after exclusion:", len(c3), "| events:", c3['aki'].sum(), f"({c3['aki'].mean():.1%})")
c3.to_csv('/content/drive/MyDrive/mimic-iv/aortic_aneurysm_final_modeling_dataset_v3.csv', index=False)

f = ('aki ~ C(approach, Treatment(reference="endovascular")) + C(location, Treatment(reference="abdominal"))'
     ' + age_at_admission + C(gender) + cci + vasopressor + C(anchor_year_group) + rupture')
m = smf.logit(f, data=c3).fit(disp=0)
print(m.summary())
ci = m.conf_int()
print(pd.DataFrame({'OR': np.exp(m.params), 'CI_low': np.exp(ci[0]), 'CI_high': np.exp(ci[1]), 'p': m.pvalues}).round(3))
X = m.model.exog
print("max VIF:", max(variance_inflation_factor(X, i) for i in range(1, X.shape[1])).round(2))

In [ ]:
base = ('aki ~ C(approach, Treatment(reference="endovascular")) + C(location, Treatment(reference="abdominal"))'
        ' + age_at_admission + C(gender) + cci + vasopressor + C(anchor_year_group)')
def run(label, rhs, data):
    m = smf.logit(base + rhs, data=data).fit(disp=0)
    k = [i for i in m.params.index if 'approach' in i][0]
    ci = np.exp(m.conf_int().loc[k])
    print(f"{label}: N={int(m.nobs)} events={int(data['aki'].sum())} open OR {np.exp(m.params[k]):.2f} ({ci[0]:.2f}-{ci[1]:.2f}) p={m.pvalues[k]:.3f}")
run("A  original",            "", c3)
run("B  + urgency",           " + C(urgency)", c3)
run("C  + rupture (primary)", " + rupture", c3)
run("D  excl. ruptured",      "", c3[c3['rupture']==0])
run("E  urgency + rupture",   " + C(urgency) + rupture", c3)

In [ ]:
cr = pd.read_csv('/content/drive/MyDrive/mimic-iv/aortic_aneurysm_creatinine_raw.csv')
icu = pd.read_csv('/content/drive/MyDrive/mimic-iv/icu/icustays.csv.gz', parse_dates=['intime'])
first_icu = icu.groupby('hadm_id')['intime'].min().rename('icu_in')

cr['admittime'] = pd.to_datetime(cr['admittime'])
cr['charttime'] = pd.to_datetime(cr['charttime'])
n0 = len(cr)
t = cr.merge(first_icu, on='hadm_id', how='left')
assert len(t) == n0, "merge changed row count"
print("rows:", n0, "| admissions:", t['hadm_id'].nunique(), "| no ICU stay found:", t['icu_in'].isna().groupby(t['hadm_id']).all().sum())

t['h_from_icu'] = (t['charttime'] - t['icu_in']).dt.total_seconds()/3600
t['h_from_adm'] = (t['charttime'] - t['admittime']).dt.total_seconds()/3600
g = t.groupby('hadm_id')

print("\nICU admit minus hospital admit (hrs):")
print(((g['icu_in'].first() - g['admittime'].first()).dt.total_seconds()/3600).describe())
print("\nshare of admissions with any creatinine BEFORE ICU admit:", (g['h_from_icu'].min() < 0).mean().round(3))
print("share whose first creatinine is >24h after hospital admit:", (g['h_from_adm'].min() > 24).mean().round(3))

In [ ]:
vc = next(c for c in ['valuenum','value','creatinine','creat'] if c in t.columns)
t = t.sort_values(['hadm_id','charttime'])
t['v'] = pd.to_numeric(t[vc], errors='coerce')

# Definition A: min creatinine within first 24h of hospital admission (fallback: first value)
def defA(df):
    w = df[df['h_from_adm'] <= 24]['v']
    base = w.min() if len(w) else df['v'].iloc[0]
    return pd.Series({'baseA': base, 'peak': df['v'].max(), 'fallback': len(w)==0})
A = t.groupby('hadm_id').apply(defA)
A['akiA'] = (A['peak'] >= 1.5*A['baseA']).astype(int)

# Definition B: running-minimum ratio (any value >= 1.5x the lowest PRIOR value)
t['prior_min'] = t.groupby('hadm_id')['v'].transform(lambda s: s.shift().cummin())
t['ratio'] = t['v'] / t['prior_min']
B = (t.groupby('hadm_id')['ratio'].max() >= 1.5).astype(int).rename('akiB')

chk = c3.merge(A[['akiA','fallback']], left_on='hadm_id', right_index=True, how='left') \
        .merge(B, left_on='hadm_id', right_index=True, how='left')
assert len(chk) == len(c3)
print("replicates saved AKI (def A vs saved):", (chk['akiA']==chk['aki']).mean().round(4))
print("fallback admissions:", int(chk['fallback'].sum()))
print("AKI rate saved / A / B:", chk['aki'].mean().round(3), chk['akiA'].mean().round(3), chk['akiB'].mean().round(3))
print("agreement saved vs B:", (chk['aki']==chk['akiB']).mean().round(4))

f = ('{y} ~ C(approach, Treatment(reference="endovascular")) + C(location, Treatment(reference="abdominal"))'
     ' + age_at_admission + C(gender) + cci + vasopressor + C(anchor_year_group) + rupture')
for y in ['aki','akiB']:
    m = smf.logit(f.format(y=y), data=chk).fit(disp=0)
    k = [i for i in m.params.index if 'approach' in i][0]
    ci = np.exp(m.conf_int().loc[k])
    print(f"{y}: open OR {np.exp(m.params[k]):.2f} ({ci[0]:.2f}-{ci[1]:.2f}) p={m.pvalues[k]:.3f}")

In [ ]:
bad = chk[chk['akiA'] != chk['aki']][['hadm_id','aki','akiA','fallback','approach','location']]
print(bad.to_string())
tt = t[t['hadm_id'].isin(bad['hadm_id'])][['hadm_id','charttime','h_from_adm','v']]
print(tt.groupby('hadm_id').head(6).to_string())

In [ ]:
fb = chk[chk['fallback']]
print("fallback admissions:", len(fb), "| saved AKI among them:", int(fb['aki'].sum()), "| def-A AKI among them:", int(fb['akiA'].sum()))
print(fb['approach'].value_counts().to_dict(), "| rupture:", int(fb['rupture'].sum()))

c4 = chk[~chk['fallback']].drop(columns=['akiA','akiB','fallback']).copy()
print("\nN:", len(c4), "| events:", int(c4['aki'].sum()), f"({c4['aki'].mean():.1%})")
c4.to_csv('/content/drive/MyDrive/mimic-iv/aortic_aneurysm_final_modeling_dataset_v4.csv', index=False)

run("A  original",            "", c4)
run("B  + urgency",           " + C(urgency)", c4)
run("C  + rupture (primary)", " + rupture", c4)
run("D  excl. ruptured",      "", c4[c4['rupture']==0])
run("E  urgency + rupture",   " + C(urgency) + rupture", c4)

m = smf.logit(f.format(y='aki'), data=c4).fit(disp=0)
ci = np.exp(m.conf_int())
print(pd.DataFrame({'OR': np.exp(m.params), 'lo': ci[0], 'hi': ci[1], 'p': m.pvalues}).round(3))

In [ ]:
import os
lab_path = EICU_PATH + 'lab.csv.gz'
if not os.path.exists(lab_path):
    lab_path = EICU_PATH + 'lab.csv'
print("using:", lab_path)

lab = pd.read_csv(lab_path, usecols=['patientunitstayid','labname','labresultoffset','labresult'])
e = pd.read_csv('/content/drive/MyDrive/eICU-CRD/aortic_aneurysm_eicu_cohort_final_v2.csv')
cr_e = lab[(lab['labname']=='creatinine') & lab['patientunitstayid'].isin(e['patientunitstayid'])].copy()
cr_e['h'] = cr_e['labresultoffset']/60
first = cr_e.groupby('patientunitstayid')['h'].min()
print("stays with creatinine:", len(first), "of", len(e))
print("first creatinine > 24h after ICU admit:", int((first > 24).sum()), f"({(first>24).mean():.1%})")
print("first creatinine < 0h (before ICU admit):", int((first < 0).sum()))

In [ ]:
c4b = c4.merge(chk[['hadm_id','akiB']], on='hadm_id', how='left')
assert len(c4b) == len(c4) and c4b['akiB'].notna().all()
print("AKI rate primary / B:", c4b['aki'].mean().round(3), c4b['akiB'].mean().round(3))
m = smf.logit(f.format(y='akiB'), data=c4b).fit(disp=0)
k = [i for i in m.params.index if 'approach' in i][0]
ci = np.exp(m.conf_int().loc[k])
print(f"Def B on v4: open OR {np.exp(m.params[k]):.2f} ({ci[0]:.2f}-{ci[1]:.2f}) p={m.pvalues[k]:.3f}")

In [ ]:
import pandas as pd, numpy as np
from google.colab import drive
drive.mount('/content/drive')

p = '/content/drive/MyDrive/mimic-iv/aortic_aneurysm_final_modeling_dataset_v4.csv'
df = pd.read_csv(p)

# --- integrity checks: print these before trusting anything ---
print("rows:", len(df), "| unique hadm_id:", df['hadm_id'].nunique())
print("AKI events:", int(df['aki'].sum()), f"({df['aki'].mean():.1%})")
print("columns:", list(df.columns))
print(df.dtypes)
print(df.isna().sum()[lambda s: s > 0])

# --- Table 1 inputs ---
print("\nlocation:\n", df['location'].value_counts(), "\n", df['location'].value_counts(normalize=True).round(3))
print("\napproach:\n", df['approach'].value_counts(), "\n", df['approach'].value_counts(normalize=True).round(3))
print("\ngender:\n", df['gender'].value_counts())
print("\nera:\n", df['anchor_year_group'].value_counts().sort_index())
for c in ['rupture', 'vasopressor']:
    if c in df: print(f"\n{c}: {int(df[c].sum())} ({df[c].mean():.1%})")
print("\nage:", df['age_at_admission'].describe().round(1).to_dict())
print("cci: mean", round(df['cci'].mean(), 2), "| median", df['cci'].median(),
      "| IQR", df['cci'].quantile([.25, .75]).tolist())
if 'urgency' in df:
    print("\nurgency:\n", df['urgency'].value_counts())

In [ ]:
import statsmodels.formula.api as smf
import numpy as np, pandas as pd

def or_table(m):
    ci = m.conf_int()
    out = pd.DataFrame({'OR': np.exp(m.params), 'lo': np.exp(ci[0]), 'hi': np.exp(ci[1]), 'p': m.pvalues})
    return out.round(3)

df['male'] = (df['gender'] == 'M').astype(int)

# 1) Reproduce primary Model C on v4
fC = ("aki ~ C(approach, Treatment('endovascular')) + C(location, Treatment('abdominal')) "
      "+ age_at_admission + C(gender) + cci + vasopressor + C(anchor_year_group) + rupture")
mC = smf.logit(fC, data=df).fit(disp=0)
print("MODEL C (v4): N =", int(mC.nobs), "| pseudo R2 =", round(mC.prsquared, 3))
print(or_table(mC))

# 2) Matched MIMIC model (abdominal + thoracic only)
dm = df[df['location'].isin(['abdominal', 'thoracic'])].copy()
dm['thoracic'] = (dm['location'] == 'thoracic').astype(int)
fM = "aki ~ thoracic + age_at_admission + male + cci + vasopressor + rupture"
mM = smf.logit(fM, data=dm).fit(disp=0)
print("\nMATCHED MIMIC: N =", int(mM.nobs), "| events =", int(dm['aki'].sum()))
print(or_table(mM))

# 3) Table 1, MIMIC column
def n_pct(mask): return f"{int(mask.sum())} ({mask.mean()*100:.1f}%)"
def med_iqr(s): return f"{s.median():.0f} ({s.quantile(.25):.0f}-{s.quantile(.75):.0f})"
rows = [
 ("N", str(len(df))),
 ("Age, median (IQR)", med_iqr(df['age_at_admission'])),
 ("Male", n_pct(df['male'] == 1)),
 ("Open repair", n_pct(df['approach'] == 'open')),
 ("Location: abdominal", n_pct(df['location'] == 'abdominal')),
 ("Location: thoracic", n_pct(df['location'] == 'thoracic')),
 ("Location: thoracoabdominal", n_pct(df['location'] == 'thoracoabdominal')),
 ("Ruptured", n_pct(df['rupture'] == 1)),
 ("Urgency: elective", n_pct(df['urgency'] == 'elective')),
 ("Urgency: emergency/urgent", n_pct(df['urgency'] == 'emergency_urgent')),
 ("Urgency: observation", n_pct(df['urgency'] == 'observation')),
 ("CCI, mean (SD)", f"{df['cci'].mean():.2f} ({df['cci'].std():.2f})"),
 ("CCI, median (IQR)", f"{df['cci'].median():.0f} ({df['cci'].quantile(.25):.0f}-{df['cci'].quantile(.75):.0f})"),
 ("Vasopressor use", n_pct(df['vasopressor'] == 1)),
 ("AKI", n_pct(df['aki'] == 1)),
]
for e in sorted(df['anchor_year_group'].unique()):
    rows.insert(-1, (f"Era {e}", n_pct(df['anchor_year_group'] == e)))
t1 = pd.DataFrame(rows, columns=['Variable', 'MIMIC-IV'])
print("\n", t1.to_string(index=False))
t1.to_csv('/content/drive/MyDrive/table1_mimic_v4.csv', index=False)

# Crude AKI by group (for Results text)
for c in ['approach', 'location', 'urgency', 'rupture']:
    print("\nAKI by", c); print(df.groupby(c)['aki'].agg(['sum', 'count', 'mean']).round(3))

In [ ]:
import scipy.stats as st
pe = '/content/drive/MyDrive/eICU-CRD/aortic_aneurysm_eicu_cohort_final_v2.csv'
ed = pd.read_csv(pe)
print("eICU rows:", len(ed), "| unique stays:", ed['patientunitstayid'].nunique(),
      "| AKI:", int(ed['aki'].sum()))
print(list(ed.columns)); print(ed.dtypes)
print(ed['gender'].value_counts(dropna=False)); print(ed['location'].value_counts(dropna=False))
print(ed.isna().sum()[lambda s: s > 0])

In [ ]:
ed['male'] = ed['gender'].astype(str).str.lower().str.startswith('m').astype(int)
ed['age_at_admission'] = ed['age_numeric']
ed['cohort'] = 1
dm['cohort'] = 0
cols = ['aki', 'thoracic', 'age_at_admission', 'male', 'cci', 'vasopressor', 'rupture', 'cohort']
ee = ed[ed['location'].isin(['abdominal', 'thoracic'])].copy()
ee['thoracic'] = (ee['location'] == 'thoracic').astype(int)
pool = pd.concat([dm[cols], ee[cols]], ignore_index=True)
print("pooled N:", len(pool), "| MIMIC", (pool.cohort == 0).sum(), "| eICU", (pool.cohort == 1).sum())

# (a) shared-covariate interaction (as before)
mA = smf.logit("aki ~ thoracic*cohort + age_at_admission + male + cci + vasopressor + rupture",
               data=pool).fit(disp=0)
print("\nSHARED-COVARIATE:"); print(or_table(mA).loc[['thoracic', 'thoracic:cohort']])

# (b) fully interacted: every covariate x cohort
cov = "thoracic + age_at_admission + male + cci + vasopressor + rupture"
mF = smf.logit(f"aki ~ ({cov})*cohort", data=pool).fit(disp=0)
mR = smf.logit(f"aki ~ {cov} + cohort", data=pool).fit(disp=0)
print("\nFULLY INTERACTED:"); print(or_table(mF))
lr = 2 * (mF.llf - mR.llf); dfree = int(mF.df_model - mR.df_model)
print(f"\nLR test, all covariate x cohort terms: chi2={lr:.2f}, df={dfree}, p={st.chi2.sf(lr, dfree):.4f}")

In [ ]:
ed['male'] = ed['gender'].astype(str).str.lower().str.startswith('m').astype(int)
def n_pct(m): return f"{int(m.sum())} ({m.mean()*100:.1f}%)"
NA = "N/A"
e = {
 "N": str(len(ed)),
 "Age, median (IQR)": f"{ed['age_numeric'].median():.0f} ({ed['age_numeric'].quantile(.25):.0f}-{ed['age_numeric'].quantile(.75):.0f})",
 "Male": n_pct(ed['male']==1),
 "Open repair": NA,
 "Location: abdominal": n_pct(ed['location']=='abdominal'),
 "Location: thoracic": n_pct(ed['location']=='thoracic'),
 "Location: thoracoabdominal": NA,
 "Ruptured": n_pct(ed['rupture']==1),
 "Urgency: elective": NA, "Urgency: emergency/urgent": NA, "Urgency: observation": NA,
 "CCI, mean (SD)": f"{ed['cci'].mean():.2f} ({ed['cci'].std():.2f})",
 "CCI, median (IQR)": f"{ed['cci'].median():.0f} ({ed['cci'].quantile(.25):.0f}-{ed['cci'].quantile(.75):.0f})",
 "Vasopressor use": n_pct(ed['vasopressor']==1),
 "AKI": n_pct(ed['aki']==1),
}
t1 = pd.read_csv('/content/drive/MyDrive/table1_mimic_v4.csv')
t1['eICU-CRD'] = t1['Variable'].map(e).fillna(NA)
extra = pd.DataFrame([["Any diagnosis data on file", NA, n_pct(ed['has_diagnosis_data']==1)]], columns=t1.columns)
t1 = pd.concat([t1, extra], ignore_index=True)
print(t1.to_string(index=False))
t1.to_csv('/content/drive/MyDrive/table1_aortic_aneurysm_v4.csv', index=False)

In [ ]:
import os, re
d = '/content/drive/MyDrive/eICU-CRD/'
files = sorted(os.listdir(d))
print(len(files), "files:"); print(files)

pat = re.compile(r'dialysis|end.?stage renal|esrd|hemodialysis|peritoneal dial', re.I)
stays = set(ed['patientunitstayid'])
hits = {}

def load(name):
    f = [x for x in files if x.lower().startswith(name.lower())]
    return pd.read_csv(d + f[0]) if f else None

ph = load('pastHistory')
if ph is not None:
    ph = ph[ph['patientunitstayid'].isin(stays)]
    txt = ph.astype(str).agg(' | '.join, axis=1)
    m = ph[txt.str.contains(pat)]
    hits['pastHistory'] = set(m['patientunitstayid'])
    print("\npastHistory: cohort stays with any row:", ph['patientunitstayid'].nunique())
    print(m.iloc[:, :6].drop_duplicates().head(15).to_string())
else:
    print("\npastHistory not found")

dx = load('diagnosis')
if dx is not None:
    dx = dx[dx['patientunitstayid'].isin(stays)]
    codes = dx['icd9code'].astype(str).str.contains(r'585\.6|N18\.6|V45\.1|Z99\.2', regex=True)
    txt = dx['diagnosisstring'].astype(str).str.contains(pat)
    hits['diagnosis'] = set(dx[codes | txt]['patientunitstayid'])
    print("\ndiagnosis hits:", len(hits['diagnosis']))
else:
    print("\ndiagnosis not found")

allhits = set().union(*hits.values()) if hits else set()
print("\nUnion of chronic-dialysis/ESRD hits:", len(allhits), "of", len(stays))
print(ed[ed['patientunitstayid'].isin(allhits)]['aki'].agg(['sum', 'count']))

In [ ]:
codes = dx['icd9code'].astype(str).str.contains(r'585\.6|N18\.6|V45\.1|Z99\.2', regex=True)
txt = dx['diagnosisstring'].astype(str).str.contains(pat)
h = dx[codes | txt]
print(h[['patientunitstayid','diagnosisstring','icd9code','diagnosisoffset']].to_string())


In [ ]:
# 1) Exclude the 2 chronic ESRD stays (1056965, 1062860)
ids = {1056965, 1062860}
e2 = ed[~ed['patientunitstayid'].isin(ids)].copy()
e2['thoracic'] = (e2['location'] == 'thoracic').astype(int)
e2['male'] = e2['gender'].astype(str).str.lower().str.startswith('m').astype(int)
e2['age_at_admission'] = e2['age_numeric']
print("eICU after exclusion:", len(e2), "| events:", int(e2['aki'].sum()), f"({e2['aki'].mean():.1%})")

# 2) eICU reduced model
mE = smf.logit("aki ~ thoracic + age_at_admission + male + cci + vasopressor + rupture", data=e2).fit(disp=0)
print("\neICU MODEL (N =", int(mE.nobs), ")"); print(or_table(mE))

# 3) Fully interacted pooled model
e2['cohort'] = 1
cols = ['aki','thoracic','age_at_admission','male','cci','vasopressor','rupture','cohort']
pool2 = pd.concat([dm[cols], e2[cols]], ignore_index=True)
cov = "thoracic + age_at_admission + male + cci + vasopressor + rupture"
mF2 = smf.logit(f"aki ~ ({cov})*cohort", data=pool2).fit(disp=0)
print("\nPOOLED N:", len(pool2)); print(or_table(mF2).loc[['thoracic','thoracic:cohort']])

# 4) Table 1 rebuilt on 382
def n_pct(m): return f"{int(m.sum())} ({m.mean()*100:.1f}%)"
NA = "N/A"
e = {
 "N": str(len(e2)),
 "Age, median (IQR)": f"{e2['age_numeric'].median():.0f} ({e2['age_numeric'].quantile(.25):.0f}-{e2['age_numeric'].quantile(.75):.0f})",
 "Male": n_pct(e2['male']==1),
 "Open repair": NA,
 "Location: abdominal": n_pct(e2['location']=='abdominal'),
 "Location: thoracic": n_pct(e2['location']=='thoracic'),
 "Location: thoracoabdominal": NA,
 "Ruptured": n_pct(e2['rupture']==1),
 "Urgency: elective": NA, "Urgency: emergency/urgent": NA, "Urgency: observation": NA,
 "CCI, mean (SD)": f"{e2['cci'].mean():.2f} ({e2['cci'].std():.2f})",
 "CCI, median (IQR)": f"{e2['cci'].median():.0f} ({e2['cci'].quantile(.25):.0f}-{e2['cci'].quantile(.75):.0f})",
 "Vasopressor use": n_pct(e2['vasopressor']==1),
 "AKI": n_pct(e2['aki']==1),
}
t1 = pd.read_csv('/content/drive/MyDrive/table1_mimic_v4.csv')
t1['eICU-CRD'] = t1['Variable'].map(e).fillna(NA)
t1.loc[t1['Variable'].str.startswith('Era'), 'eICU-CRD'] = "2014-2015 (single window)"
extra = pd.DataFrame([["Any diagnosis data on file", NA, n_pct(e2['has_diagnosis_data']==1)]], columns=t1.columns)
t1 = pd.concat([t1, extra], ignore_index=True)
print("\n", t1.to_string(index=False))
t1.to_csv('/content/drive/MyDrive/table1_aortic_aneurysm_v5.csv', index=False)

In [ ]:
d = '/content/drive/MyDrive/eICU-CRD/'
lab = pd.read_csv(d + 'lab.csv.gz', usecols=['patientunitstayid','labresultoffset','labname','labresult'])
lab = lab[(lab['labname'].str.lower() == 'creatinine') & lab['patientunitstayid'].isin(e2['patientunitstayid'])]
pt = pd.read_csv(d + 'patient.csv.gz', usecols=['patientunitstayid','unitdischargeoffset'])
lab = lab.merge(pt, on='patientunitstayid', how='left')
lab = lab[lab['labresultoffset'] <= lab['unitdischargeoffset']].dropna(subset=['labresult'])
print("stays with creatinine:", lab['patientunitstayid'].nunique(), "(expect 382)")

def aki_def(df, allow_pre):
    x = df if allow_pre else df[df['labresultoffset'] >= 0]
    base = x[x['labresultoffset'] <= 1440].groupby('patientunitstayid')['labresult'].min().rename('base')
    peak = x.groupby('patientunitstayid')['labresult'].max().rename('peak')
    r = pd.concat([base, peak], axis=1)
    r['aki'] = (r['peak'] >= 1.5 * r['base']).astype(float)
    r.loc[r['base'].isna(), 'aki'] = np.nan
    return r['aki']

cmp = e2[['patientunitstayid','aki']].set_index('patientunitstayid')
cmp['def1_with_pre'] = aki_def(lab, True)
cmp['def2_no_pre'] = aki_def(lab, False)
print(cmp.isna().sum())
for c in ['def1_with_pre','def2_no_pre']:
    ok = cmp[c].notna()
    print(f"\n{c}: AKI {cmp.loc[ok,c].mean():.1%} (n={ok.sum()}) | agrees with saved aki: {(cmp.loc[ok,c]==cmp.loc[ok,'aki']).mean():.1%}")
    print(pd.crosstab(cmp.loc[ok,'aki'], cmp.loc[ok,c]))

In [ ]:
lab0 = pd.read_csv(d + 'lab.csv.gz', usecols=['patientunitstayid','labresultoffset','labname','labresult'])
lab0 = lab0[(lab0['labname'].str.lower()=='creatinine') & lab0['patientunitstayid'].isin(e2['patientunitstayid'])].dropna(subset=['labresult'])

# Variant A: Def 1 but no discharge-offset cutoff on the peak
cmp['A_no_dc_cut'] = aki_def(lab0, True)

# Variant B: running minimum (any value >= 1.5x lowest prior value), whole stay
def runmin_aki(df):
    out = {}
    for sid, g in df.sort_values('labresultoffset').groupby('patientunitstayid'):
        v = g['labresult'].values
        prior_min = np.minimum.accumulate(v)
        out[sid] = float(any(v[i] >= 1.5 * prior_min[i-1] for i in range(1, len(v))))
    return pd.Series(out)
cmp['B_runmin'] = runmin_aki(lab0)

for c in ['A_no_dc_cut','B_runmin']:
    ok = cmp[c].notna()
    print(f"\n{c}: AKI {cmp.loc[ok,c].mean():.1%} | agrees with saved: {(cmp.loc[ok,c]==cmp.loc[ok,'aki']).mean():.1%}")
    print(pd.crosstab(cmp.loc[ok,'aki'], cmp.loc[ok,c]))

# The 8 stays: saved aki=1 but Def 1 = 0
bad = cmp[(cmp['aki']==1) & (cmp['def1_with_pre']==0)].index
print("\nMismatch stays:", list(bad))
print(lab0[lab0['patientunitstayid'].isin(bad)]
      .sort_values(['patientunitstayid','labresultoffset'])
      [['patientunitstayid','labresultoffset','labresult']].to_string(index=False))
print(pt[pt['patientunitstayid'].isin(bad)].to_string(index=False))

In [ ]:
import scipy.stats as st

# 1) Table 3: p-values for models A, B, E (v4, N=740)
base = ("aki ~ C(approach, Treatment('endovascular')) + C(location, Treatment('abdominal')) "
        "+ age_at_admission + C(gender) + cci + vasopressor + C(anchor_year_group)")
forms = {"A base": base, "B +urgency": base + " + C(urgency)",
         "E +urgency+rupture": base + " + C(urgency) + rupture"}
key = "C(approach, Treatment('endovascular'))[T.open]"
for k, f in forms.items():
    m = smf.logit(f, data=df).fit(disp=0)
    ci = np.exp(m.conf_int().loc[key])
    print(f"{k}: N={int(m.nobs)} OR={np.exp(m.params[key]):.2f} ({ci[0]:.2f}-{ci[1]:.2f}) p={m.pvalues[key]:.4f}")

# 2) Cardiac-surgery check on the 382 cohort
tr = pd.read_csv('/content/drive/MyDrive/eICU-CRD/treatment.csv.gz',
                 usecols=['patientunitstayid', 'treatmentstring'])
tr = tr[tr['patientunitstayid'].isin(e2['patientunitstayid'])]
cs = set(tr[tr['treatmentstring'].str.contains('cardiovascular|cardiac surgery', case=False, regex=False)]['patientunitstayid'])
e2['cardsurg'] = e2['patientunitstayid'].isin(cs).astype(int)
print("\ncardiac-surgery-type treatment by location (share):")
print(e2.groupby('location')['cardsurg'].agg(['sum', 'count', 'mean']).round(3))
e3 = e2[e2['cardsurg'] == 0].copy()
print("after exclusion: N =", len(e3), "| events =", int(e3['aki'].sum()), "| excluded =", int(e2['cardsurg'].sum()))
cov = "thoracic + age_at_admission + male + cci + vasopressor + rupture"
m3 = smf.logit(f"aki ~ {cov}", data=e3).fit(disp=0)
print(or_table(m3).loc[['thoracic']])
cols = ['aki', 'thoracic', 'age_at_admission', 'male', 'cci', 'vasopressor', 'rupture', 'cohort']
p3 = pd.concat([dm[cols], e3[cols]], ignore_index=True)
mF3 = smf.logit(f"aki ~ ({cov})*cohort", data=p3).fit(disp=0)
print(or_table(mF3).loc[['thoracic', 'thoracic:cohort']])

# 3) Per-covariate interactions on the 382 pooled model, plus 5-df test excluding location
print("\nFULL INTERACTED (382 cohort):")
print(or_table(mF2).loc[[i for i in mF2.params.index if ':cohort' in i]])
mR2 = smf.logit("aki ~ thoracic*cohort + age_at_admission + male + cci + vasopressor + rupture", data=pool2).fit(disp=0)
lr5 = 2 * (mF2.llf - mR2.llf); d5 = int(round(mF2.df_model - mR2.df_model))
print(f"5-df LR (covariates other than location x cohort): chi2={lr5:.2f}, df={d5}, p={st.chi2.sf(lr5, d5):.4f}")

In [ ]:
import os, pandas as pd
root = '/content/drive/MyDrive/mimic-iv/'
hosp = root + 'hosp/'

# 1) version check: anything that names it?
print("top level:", os.listdir(root))
for n in ['README.md', 'LICENSE.txt', 'SHA256SUMS.txt']:
    if os.path.exists(root + n): print(f"\n--- {n} (first 500 chars) ---\n", open(root + n).read()[:500])

def rd(n, **k):
    for ext in ['.csv.gz', '.csv']:
        if os.path.exists(hosp + n + ext): return pd.read_csv(hosp + n + ext, **k)
    print("missing:", n)

ids = set(df['hadm_id'])

# 2) aortic procedure codes in the cohort
pr = rd('procedures_icd', usecols=['hadm_id', 'icd_code', 'icd_version'])
pr = pr[pr['hadm_id'].isin(ids)]
dpr = rd('d_icd_procedures')
t = pr.merge(dpr, on=['icd_code', 'icd_version'], how='left')
icd9_core = {'3844', '3971', '3973', '3978'}
keep = t['long_title'].str.contains('aort', case=False, na=False) | t['icd_code'].isin(icd9_core)
s = (t[keep].groupby(['icd_version', 'icd_code', 'long_title'])['hadm_id']
     .nunique().reset_index(name='admissions').sort_values(['icd_version', 'admissions'], ascending=[True, False]))
print("\nAORTIC PROCEDURE CODES IN COHORT:\n", s.to_string(index=False))
s.to_csv('/content/drive/MyDrive/supp_procedure_codes.csv', index=False)

# 3) rupture and ESRD diagnosis codes present in the cohort
dx = rd('diagnoses_icd', usecols=['hadm_id', 'icd_code', 'icd_version'])
dx = dx[dx['hadm_id'].isin(ids)]
ddx = rd('d_icd_diagnoses')
d = dx.merge(ddx, on=['icd_code', 'icd_version'], how='left')
tt = d['long_title'].str.lower().fillna('')
rup = d[tt.str.contains('aneurysm') & tt.str.contains('ruptured') & ~tt.str.contains('nonruptured|without rupture|cerebral|intracranial')]
esrd_core = {'5856', 'N186', 'V451', 'V4511', 'Z992', '40301', '40311', '40391', '40402', '40403',
             '40412', '40413', '40492', '40493', 'I120', 'I1311', 'I132'}
esr = d[d['icd_code'].isin(esrd_core)]
for name, x in [('RUPTURE', rup), ('ESRD (in v4 cohort, expect ~0 since excluded)', esr)]:
    out = (x.groupby(['icd_version', 'icd_code', 'long_title'])['hadm_id'].nunique()
           .reset_index(name='admissions'))
    print(f"\n{name}:\n", out.to_string(index=False))
    out.to_csv(f"/content/drive/MyDrive/supp_{name.split()[0].lower()}_codes.csv", index=False)

In [ ]:
import os
hosp = '/content/drive/MyDrive/mimic-iv/hosp/'
def rd(n, **k):
    for ext in ['.csv.gz', '.csv']:
        if os.path.exists(hosp + n + ext): return pd.read_csv(hosp + n + ext, **k)
pr = rd('procedures_icd', usecols=['hadm_id', 'icd_code', 'icd_version'])
pr = pr[pr['hadm_id'].isin(set(df['hadm_id']))]
t = pr.merge(rd('d_icd_procedures'), on=['icd_code', 'icd_version'], how='left')
ttl = t['long_title'].str.lower().fillna('')
t['valve'] = ttl.str.contains('aortic valve')
t['cabg'] = ttl.str.contains('coronary')
t['asc'] = ttl.str.contains('ascending/arch')

# 1) concomitant cardiac procedures
fl = t.groupby('hadm_id')[['valve', 'cabg', 'asc']].max()
d2 = df.merge(fl, left_on='hadm_id', right_index=True, how='left')
for c in ['valve', 'cabg', 'asc']: d2[c] = d2[c].fillna(False).astype(int)
d2['concom'] = d2[['valve', 'cabg', 'asc']].max(axis=1)
print("rows:", len(d2), "(must be 740)")
print(d2[['valve', 'cabg', 'asc', 'concom']].sum())
print(d2.groupby('concom')['aki'].agg(['sum', 'count', 'mean']).round(3))
print(pd.crosstab(d2['location'], d2['concom'])); print(pd.crosstab(d2['approach'], d2['concom']))
fC = ("aki ~ C(approach, Treatment('endovascular')) + C(location, Treatment('abdominal')) "
      "+ age_at_admission + C(gender) + cci + vasopressor + C(anchor_year_group) + rupture")
d3 = d2[d2['concom'] == 0]
m = smf.logit(fC, data=d3).fit(disp=0)
print("\nPRIMARY MODEL EXCLUDING CONCOMITANT:", int(m.nobs), "| events", int(d3['aki'].sum()))
print(or_table(m))

# 2) approach label vs PCS approach character (5th char: 0 = open, 3 = percutaneous)
c = t.merge(df[['hadm_id', 'approach']], on='hadm_id')
c = c[(c['icd_version'] == 10) & c['icd_code'].str.match(r'^(04V|04R|02VW|02RW)')]
c['root'] = c['icd_code'].str[:3] + c['icd_code'].str[3:4]
c['pcs_approach'] = c['icd_code'].str[4]
print("\nAPPROACH LABEL vs PCS CODE (admissions):")
print(c.groupby(['root', 'pcs_approach', 'approach'])['hadm_id'].nunique().unstack(fill_value=0))

In [ ]:
ttl = t['long_title'].str.lower().fillna('')
diag = 'ultrasonography|fluoroscopy|radiography|measurement|monitoring|angiography|imaging'
t['valve'] = ttl.str.contains('aortic valve') & ~ttl.str.contains(diag)
t['cabg'] = ttl.str.contains('bypass coronary|coronary bypass|coronary artery bypass')
t['asc'] = ttl.str.contains('ascending/arch')
t['cpb'] = t['icd_code'].isin(['3961', '5A1221Z'])

# what the broad 'coronary' flag was catching that the strict one is not
broad_only = t[ttl.str.contains('coronary') & ~t['cabg']]
print("coronary titles NOT counted as CABG:\n", broad_only.groupby('long_title')['hadm_id'].nunique().sort_values(ascending=False).head(10))
print("\nCPB-code titles:", t.loc[t['cpb'], 'long_title'].unique())

fl = t.groupby('hadm_id')[['valve', 'cabg', 'asc', 'cpb']].max()
s = df.merge(fl, left_on='hadm_id', right_index=True, how='left')
for c in ['valve', 'cabg', 'asc', 'cpb']: s[c] = s[c].fillna(False).astype(int)
s['concom'] = s[['valve', 'cabg', 'asc', 'cpb']].max(axis=1)
print("\nrows:", len(s), "(must be 740)"); print(s[['valve', 'cabg', 'asc', 'cpb', 'concom']].sum())
print(s.groupby('concom')['aki'].agg(['sum', 'count', 'mean']).round(3))
print(pd.crosstab(s['location'], s['concom']))

keyo = "C(approach, Treatment('endovascular'))[T.open]"
keyt = "C(location, Treatment('abdominal'))[T.thoracic]"
m1 = smf.logit(fC, data=s[s.concom == 0]).fit(disp=0)
m2 = smf.logit(fC + " + concom", data=s).fit(disp=0)
for name, m in [("EXCLUDING concomitant (strict)", m1), ("ADJUSTED for concomitant, N=740", m2)]:
    o = or_table(m)
    print(f"\n{name}: N={int(m.nobs)}"); print(o.loc[[keyo, keyt]])
print("concom OR in adjusted model:", or_table(m2).loc['concom'].to_dict())

In [ ]:
s['cardiac'] = s[['valve', 'cabg', 'asc']].max(axis=1)
print("cardiac-surgery flag (valve/CABG/ascending):", int(s['cardiac'].sum()), "of", len(s))
print(pd.crosstab(s['cardiac'], s['cpb'], margins=True))
print(s.groupby(['cardiac', 'cpb'])['aki'].agg(['sum', 'count', 'mean']).round(3))
print(pd.crosstab(s['location'], [s['cardiac'], s['cpb']]))

keyo = "C(approach, Treatment('endovascular'))[T.open]"
keyt = "C(location, Treatment('abdominal'))[T.thoracic]"
mG = smf.logit(fC, data=s[s['cardiac'] == 0]).fit(disp=0)
mH = smf.logit(fC + " + cardiac + cpb", data=s).fit(disp=0)
for name, m in [("G: EXCLUDING valve/CABG/ascending", mG), ("H: ADJUSTED for cardiac and cpb, N=740", mH)]:
    print(f"\n{name}: N={int(m.nobs)}"); print(or_table(m).loc[[keyo, keyt]])
print(or_table(mH).loc[['cardiac', 'cpb']])

In [ ]:
from scipy import stats as st
def pf(p): return "<0.001" if p < 0.001 else f"{p:.3f}"
def cont(d, c, lab):
    a, b = d[d.aki == 1][c], d[d.aki == 0][c]
    f = lambda s: f"{s.median():.0f} ({s.quantile(.25):.0f}-{s.quantile(.75):.0f})"
    return [[f"{lab}, median (IQR)", f(d[c]), f(a), f(b), pf(st.mannwhitneyu(a, b).pvalue)]]
def cat(d, c, lab):
    ct = pd.crosstab(d[c], d['aki'])
    p = st.fisher_exact(ct)[1] if ct.shape == (2, 2) else st.chi2_contingency(ct)[1]
    na, nb = d.aki.sum(), (1 - d.aki).sum(); rows = []
    for i, lv in enumerate(ct.index):
        n1, n0 = ct.loc[lv].get(1, 0), ct.loc[lv].get(0, 0)
        rows.append([f"{lab}: {lv}", f"{n1+n0} ({(n1+n0)/len(d)*100:.1f}%)", f"{n1} ({n1/na*100:.1f}%)",
                     f"{n0} ({n0/nb*100:.1f}%)", pf(p) if i == 0 else ""])
    return rows
def build(d, spec):
    rows = [["N", str(len(d)), str(int(d.aki.sum())), str(int((1 - d.aki).sum())), ""]]
    for kind, c, lab in spec: rows += (cont if kind == 'c' else cat)(d, c, lab)
    return pd.DataFrame(rows, columns=['Variable', 'All', 'AKI', 'No AKI', 'p'])

d = df.copy(); d['male'] = (d.gender == 'M').astype(int)
tm = build(d, [('c','age_at_admission','Age'), ('c','cci','CCI'), ('k','male','Male'), ('k','approach','Approach'),
               ('k','location','Location'), ('k','rupture','Ruptured'), ('k','urgency','Urgency'),
               ('k','vasopressor','Vasopressor'), ('k','anchor_year_group','Era')])
e = e2.copy(); e['male'] = e['gender'].astype(str).str.lower().str.startswith('m').astype(int)
te = build(e, [('c','age_numeric','Age'), ('c','cci','CCI'), ('k','male','Male'), ('k','location','Location'),
               ('k','rupture','Ruptured'), ('k','vasopressor','Vasopressor')])
print("MIMIC-IV\n", tm.to_string(index=False)); print("\neICU-CRD\n", te.to_string(index=False))
tm.to_csv('/content/drive/MyDrive/table1_mimic_by_aki.csv', index=False)
te.to_csv('/content/drive/MyDrive/table1_eicu_by_aki.csv', index=False)

In [ ]:
!pip -q install comorbidipy polars
import sys, importlib.metadata as m
print("python", sys.version.split()[0])
for p in ["pandas", "numpy", "polars", "statsmodels", "scipy", "comorbidipy"]:
    print(p, m.version(p))

In [ ]:
  import sys, importlib.metadata as m
  print("python", sys.version.split()[0])
  for p in ["pandas", "numpy", "polars", "statsmodels", "scipy", "comorbidipy"]:
      print(p, m.version(p))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob
for p in glob.glob('/content/drive/MyDrive/**/*.csv', recursive=True):
    if any(k in p.lower() for k in ['aneurysm','aortic','modeling','cohort']):
        print(p)

In [ ]:
import pandas as pd, numpy as np
import statsmodels.formula.api as smf

df = pd.read_csv('/content/drive/MyDrive/mimic-iv/aortic_aneurysm_final_modeling_dataset_v4.csv')
print(df.shape)
print(df.columns.tolist())
print(df['aki'].sum() if 'aki' in df else 'no aki column')

In [ ]:
print(df['approach'].value_counts(dropna=False))
print(df['location'].value_counts(dropna=False))
print(df['urgency'].value_counts(dropna=False))

f = ("aki ~ C(approach, Treatment(reference='endovascular')) + C(location, Treatment(reference='abdominal'))"
     " + age_at_admission + C(gender) + cci + vasopressor + C(anchor_year_group) + rupture")
res = smf.logit(f, data=df).fit(disp=0)
print(int(res.nobs), int(df['aki'].sum()))
t = np.exp(res.conf_int()); t.columns = ['lo','hi']
t['OR'] = np.exp(res.params); t['p'] = res.pvalues
print(t.round(3))

print(pd.crosstab(df['urgency'], df['approach'], margins=True))
print(pd.crosstab(df['urgency'], df['approach'], normalize='index').round(3))

In [ ]:
print(np.exp(res.conf_int().iloc[1]).values, np.exp(res.params.iloc[1]))

In [ ]:
import glob
print(glob.glob('/content/drive/MyDrive/**/supp_procedure_codes*.csv', recursive=True))

In [ ]:
codes = pd.read_csv('/content/drive/MyDrive/supp_procedure_codes.csv')
print(codes.shape)
print(codes.columns.tolist())
print(codes.head(40).to_string())

In [ ]:
LABS = LABS.drop(columns=['admittime','dischtime'])   # use the parsed ADM versions instead
print(LABS.itemid.value_counts().head())               # expect 50912 only (or mostly)

In [ ]:
import pandas as pd
import os
from google.colab import drive

# Ensure Google Drive is mounted
drive.mount('/content/drive', force_remount=True)

# Verified paths from Google Drive glob
P = "/content/drive/MyDrive/mimic-iv/"
HOSP_P = "/content/drive/MyDrive/mimic-iv/hosp/"

COHORT = pd.read_csv(os.path.join(P, "aortic_aneurysm_final_modeling_dataset_v4.csv"))
ADM    = pd.read_csv(os.path.join(HOSP_P, "admissions.csv.gz"), usecols=['hadm_id','admittime','dischtime'],
                     parse_dates=['admittime','dischtime'])
LABS   = pd.read_csv(os.path.join(P, "aortic_aneurysm_creatinine_raw.csv"), parse_dates=['charttime'])

# make the file fit the cell: add itemid if the file is already creatinine-only
if 'itemid' not in LABS.columns:
    LABS['itemid'] = 50912
CR_ITEM = 50912

# quick sanity checks before running the main cell
print(COHORT.shape, COHORT.hadm_id.nunique())          # expect 740 and 740
print(LABS.columns.tolist(), LABS.hadm_id.nunique())
print(set(COHORT.columns) >= {'aki','approach','location','age_at_admission','gender',
                              'cci','vasopressor','anchor_year_group','rupture'})   # expect True

In [ ]:
import numpy as np, statsmodels.formula.api as smf, statsmodels.api as sm

FORMULA = ("aki7 ~ C(approach, Treatment('endovascular')) + C(location, Treatment('abdominal'))"
           " + age_at_admission + C(gender) + cci + vasopressor + C(anchor_year_group) + rupture")

cr = (LABS[(LABS.itemid == CR_ITEM) & LABS.hadm_id.isin(COHORT.hadm_id)]
        [['hadm_id','charttime','valuenum']].dropna()
        .merge(ADM[['hadm_id','admittime','dischtime']], on='hadm_id'))
cr['charttime'] = pd.to_datetime(cr.charttime)
cr = cr[(cr.charttime >= cr.admittime) & (cr.charttime <= cr.dischtime)]
cr = cr.sort_values(['hadm_id','charttime'])

# Version B (KDIGO-faithful): each creatinine vs the lowest EARLIER value within the prior 7 days
def prior7_min(g):
    s = g.set_index('charttime')['valuenum']
    return s.rolling('7D', closed='left').min().values
cr['prior_min'] = np.concatenate([prior7_min(g) for _, g in cr.groupby('hadm_id', sort=False)])
cr['ratio7'] = cr.valuenum / cr.prior_min
aki_B = (cr.groupby('hadm_id').ratio7.max() >= 1.5).rename('aki7')

# Version A (simple): primary baseline (lowest in first 24 h), peak only within 7 days of admission
first24 = cr[cr.charttime <= cr.admittime + pd.Timedelta('24h')]
base = first24.groupby('hadm_id').valuenum.min().rename('base')
w7 = cr[cr.charttime <= cr.admittime + pd.Timedelta('7D')]
peak7 = w7.groupby('hadm_id').valuenum.max().rename('peak7')
aki_A = ((peak7 / base) >= 1.5).rename('aki7A')

d = COHORT.merge(aki_B, on='hadm_id', how='left').merge(aki_A, on='hadm_id', how='left')
d['aki7']  = d.aki7.fillna(False).astype(int)
d['aki7A'] = d.aki7A.fillna(False).astype(int)

print("Crude AKI, primary:", round(d.aki.mean(),3),
      "| B (rolling 7d):", round(d.aki7.mean(),3),
      "| A (peak <=7d of admission):", round(d.aki7A.mean(),3))
print("Primary-positive but B-negative:", ((d.aki==1)&(d.aki7==0)).sum())

for label, y in [("B: KDIGO rolling 7-day", "aki7"), ("A: peak within 7d of admission", "aki7A")]:
    m = smf.glm(FORMULA.replace("aki7 ~", f"{y} ~"), data=d, family=sm.families.Binomial()).fit()
    term = [t for t in m.params.index if 'approach' in t][0]
    or_, (lo, hi) = np.exp(m.params[term]), np.exp(m.conf_int().loc[term])
    print(f"{label}: events={int(d[y].sum())}, open vs endo OR {or_:.2f} ({lo:.2f} to {hi:.2f}), p={m.pvalues[term]:.3f}")